# Livrable 2 — Débruitage de photographies par auto-encodeur convolutif

Ce notebook reprend la démarche du workshop *Auto-encodeur convolutif et traitement d'images*, appliquée cette fois aux **photographies du projet TouNum** au lieu de MNIST.

Le livrable 1 a permis d'isoler les photographies parmi les documents numérisés. Ces photos sont souvent dégradées par la numérisation (bruit du capteur, grain du papier...). L'objectif est de les **débruiter** avant l'étape de génération de légendes :

**Image propre → ajout de bruit → image bruitée → auto-encodeur → image reconstruite**

Par rapport au workshop, plusieurs choses changent :

| | Workshop | Livrable 2 |
| --- | --- | --- |
| Données | MNIST, 70 000 images, chargé avec `keras.datasets` | **148 photos** du dossier `datasets/Livrable 2` |
| Couleurs | niveaux de gris, 1 canal | **RGB, 3 canaux** |
| Taille | 28×28 | patchs 128×128 pour l'entraînement, images 256×256 pour le test |

# 1. Traitement d'images

## 1.1 Importation des librairies

In [ ]:
import os
import pathlib
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm

#pour pouvoir entraîner avec le gpu (même configuration que le livrable 1)
os.environ['KERAS_BACKEND'] = 'torch'
import keras
import tensorflow as tf

In [ ]:
import torch.cuda as cuda

print("Active backend:", keras.backend.backend())
print(cuda.is_available())
if cuda.is_available():
    print(cuda.get_device_name(0))

## 1.2 Dataset

Le dataset fourni pour ce livrable se trouve dans le dossier `datasets/Livrable 2` : **148 photographies** JPEG de tailles variées (640×480, 500×375, formats portrait et paysage...).

Les fichiers s'appellent `noisy_XXX.jpg`, mais en pratique le bruit qu'ils contiennent est **faible** (léger grain et artefacts de compression JPEG). Surtout, on ne dispose d'**aucune version propre** de ces images : impossible donc d'entraîner directement le modèle à les « nettoyer », puisqu'il n'y a pas de cible. On applique donc la démarche demandée : ces photos servent d'**images de référence**, et on leur ajoute un **bruit contrôlé** pour créer les paires (image bruitée, image propre). On reviendra en fin de notebook (section 2.4) sur l'effet du modèle sur le bruit d'origine des fichiers.

Dans le workshop, MNIST était chargé directement avec `keras.datasets.mnist.load_data()`, déjà découpé en train / test et au format numpy. Ici, il faut construire nous-mêmes ces tableaux à partir des fichiers.

**Prise en compte des images RGB.** Une image MNIST est une matrice 28×28 (un seul canal), d'où le `reshape` en `(28, 28, 1)` dans le workshop. Une photographie couleur est composée de **3 canaux (rouge, vert, bleu)** : une image a donc la forme `(hauteur, largeur, 3)`. Toutes les images du dossier sont déjà en RGB, mais on les convertit tout de même explicitement (`.convert("RGB")`) : une image en niveaux de gris ou avec canal de transparence ajoutée plus tard au dossier serait ainsi ramenée au même format, sans quoi les images ne pourraient pas être réunies dans un même tableau.

**Redimensionnement.** Les photos ont des formats variés. On les recadre au centre en carré (pour ne pas les déformer), puis on les redimensionne en 256×256.

**Normalisation.** Comme dans le workshop, les pixels sont divisés par 255 pour être ramenés entre 0 et 1.

In [ ]:
# Configurations principales de nos modèles
LOAD_SIZE         = 256                               # taille coté des images chargées (256x256)
IMG_SIZE          = 128                               # taille coté des patchs d'entraînement (128x128)
PATCHES_PER_IMAGE = 16                                # nombre de patchs extraits par image d'entraînement
NB_EPOCHS_DENOISE = 50                                # nombre epoch algorithme debruiter
BATCH_SIZE        = 32                                # taille batch de traitement
SAV_MODEL_DENOISE = "denoiser_livrable2.keras"   # sauvegarde du modele de debruitage

photo_dir = pathlib.Path('datasets') / 'Livrable 2'
np.random.seed(42)
keras.utils.set_random_seed(42)

In [ ]:
def load_photo(path):
    with Image.open(path) as img:
        img = img.convert("RGB")
        w, h = img.size
        side = min(w, h)
        left, top = (w - side) // 2, (h - side) // 2
        img = img.crop((left, top, left + side, top + side)).resize((LOAD_SIZE, LOAD_SIZE), Image.BICUBIC)
        return np.asarray(img)


photo_paths = sorted(p for p in photo_dir.iterdir() if p.is_file())
photo_paths = [photo_paths[i] for i in np.random.permutation(len(photo_paths))]

images = []
for path in tqdm(photo_paths, desc="Chargement des photos"):
    try:
        images.append(load_photo(path))
    except OSError as e:
        print(f"Fichier corrompu ignoré : {path} ({e})")
images = np.stack(images)

# découpage 70 % train / 15 % validation / 15 % test, par image
n_train, n_val = int(0.7 * len(images)), int(0.15 * len(images))
x_train_full = images[:n_train]
x_val = images[n_train:n_train + n_val]
x_test = images[n_train + n_val:]
print(f"train : {len(x_train_full)} images  |  validation : {len(x_val)}  |  test : {len(x_test)}")

### Data augmentation : découpage en patchs

Avec seulement ~100 images d'entraînement, le modèle verrait trop peu d'exemples (le workshop en avait 60 000). Le découpage est fait **par image** : une même photo ne peut pas se retrouver à la fois dans l'entraînement et dans le test.

Pour augmenter le nombre d'exemples, on extrait de chaque image d'entraînement **16 patchs de 128×128 à des positions aléatoires**, retournés horizontalement une fois sur deux. Le modèle apprend ainsi sur ~1 600 exemples variés. C'est une data augmentation simple, adaptée au débruitage : le bruit étant un phénomène **local** (pixel par pixel), le modèle n'a pas besoin de voir l'image entière pour apprendre à le retirer.

Comme l'auto-encodeur n'est composé que de couches de convolution et de pooling, il peut ensuite traiter des images de **n'importe quelle taille** (multiple de 4) : la validation et le test se font donc sur les images entières en 256×256.

In [ ]:
def extract_patches(imgs, n_patches=PATCHES_PER_IMAGE, size=IMG_SIZE):
    patches = []
    for img in imgs:
        for _ in range(n_patches):
            top, left = np.random.randint(0, img.shape[0] - size + 1, size=2)
            patch = img[top:top + size, left:left + size]
            patches.append(patch[:, ::-1] if np.random.rand() < 0.5 else patch)
    return np.stack(patches)


x_train = extract_patches(x_train_full)

In [ ]:
# Data pre-processing
x_train = x_train.astype('float32') / 255.
x_val = x_val.astype('float32') / 255.
x_test = x_test.astype('float32') / 255.

# plus besoin de reshape comme pour MNIST : les images ont déjà leurs 3 canaux
print(x_train.shape, x_val.shape, x_test.shape)

Dans le workshop, le jeu de test servait aussi de validation pendant l'entraînement. Ici, on garde un jeu de **validation** séparé pour suivre l'entraînement, et le jeu de **test** n'est utilisé qu'à la fin pour l'étude de cas : les résultats présentés ne sont donc pas biaisés par l'entraînement.

La fonction d'affichage du workshop utilisait `cmap='gray'`. Pour des images RGB, `imshow` affiche directement les couleurs : il suffit de retirer le `cmap`. On l'étend aussi pour pouvoir afficher plusieurs séries d'images l'une sous l'autre (originales, bruitées, débruitées).

In [ ]:
def display_image(*rows, n=8, titles=None):
    plt.figure(figsize=(2 * n, 2.2 * len(rows)))
    for r, X in enumerate(rows):
        for i in range(n):
            ax = plt.subplot(len(rows), n, r * n + i + 1)
            plt.imshow(np.clip(X[i], 0, 1))
            ax.get_xaxis().set_visible(False)
            ax.get_yaxis().set_visible(False)
            if titles and i == 0:
                ax.set_title(titles[r], loc="left")
    plt.show()

Quelques images du dataset, puis quelques patchs d'entraînement :

In [ ]:
display_image(x_test, n=8)
display_image(x_train, n=8)

## 1.3 Auto-encodeur de réduction de bruit (denoiser)

**Quelles sont les données d'entrée et de sortie de l'auto-encodeur ?**

* **entrée** : l'image **bruitée** `x_train_noisy` ;
* **sortie attendue (cible)** : l'image **propre** d'origine `x_train`.

Le modèle apprend donc à passer d'une image bruitée à l'image propre. C'est la différence avec l'auto-encodeur de réduction de dimension, où l'entrée et la sortie sont la même image.

### Ajout contrôlé de bruit

Comme dans le workshop, le dataset ne contient pas d'images bruitées : on génère un **bruit gaussien** avec `np.random.normal`, puis on le `clip` pour que les pixels restent entre 0 et 1.

Le `noise_factor` de 0.5 du workshop convient pour MNIST (chiffres blancs sur fond noir, très contrastés), mais sur des photographies il détruit presque toute l'image. On utilise donc **`noise_factor = 0.1`** : le bruit est bien visible, sans rendre la photo méconnaissable. Le bruit est ajouté indépendamment sur chacun des 3 canaux.

In [ ]:
noise_factor = 0.5
x_train_noisy = x_train + np.random.normal(loc=0.0, scale=1.0, size=x_train.shape) * noise_factor
x_val_noisy = x_val + np.random.normal(loc=0.0, scale=1.0, size=x_val.shape) * noise_factor
x_test_noisy = x_test + np.random.normal(loc=0.0, scale=1.0, size=x_test.shape) * noise_factor

x_train_noisy = np.clip(x_train_noisy, 0., 1.).astype('float32')
x_val_noisy = np.clip(x_val_noisy, 0., 1.).astype('float32')
x_test_noisy = np.clip(x_test_noisy, 0., 1.).astype('float32')

Visualisons les images d'origine puis les images bruitées :

In [ ]:
display_image(x_train, x_train_noisy, n=8, titles=["Originales", "Bruitées"])

###### Encodeur

On reprend l'architecture du workshop : des couches `Conv2D` qui extraient les caractéristiques de l'image, suivies de `MaxPooling2D` qui divisent la résolution par 2. Toutes les convolutions utilisent `padding='same'` pour que seuls les poolings modifient la taille.

Adaptations pour les photos :

* l'entrée a 3 canaux au lieu d'1. Sa hauteur et sa largeur sont laissées libres (`None`) pour pouvoir entraîner sur des patchs 128×128 puis tester sur des images 256×256 ;
* les photos sont beaucoup plus riches que des chiffres manuscrits (couleurs, textures) : on utilise 32 puis 64 filtres pour avoir assez de capacité.

Après deux poolings, la résolution est divisée par 4 : un patch 128×128 est encodé en une représentation `32×32×64`.

In [ ]:
from keras.layers import Input, Conv2D, MaxPooling2D, UpSampling2D


# The encoding process
# (None, None) : le modèle accepte des images de n'importe quelle taille (patchs 128x128 ou images 256x256)
input_img = Input(shape=(None, None, 3))

# Encoding #

# Conv1 #
x = Conv2D(32, (3, 3), activation='relu', padding='same')(input_img)
x = MaxPooling2D((2, 2), padding='same')(x)

# Conv 2 #
x = Conv2D(64, (3, 3), activation='relu', padding='same')(x)
encoded = MaxPooling2D((2, 2), padding='same')(x)

###### Décodeur

Le décodeur est l'inverse de l'encodeur : des `Conv2D` suivies de `UpSampling2D` qui doublent la résolution, jusqu'à retrouver la taille d'entrée.

La dernière couche est la principale adaptation au RGB : elle produit **3 canaux** (au lieu d'1 pour MNIST), avec une activation **sigmoïde** pour que chaque pixel reconstruit soit compris entre 0 et 1, comme l'image cible.

In [ ]:
# Decoding #

# DeConv1
x = Conv2D(64, (3, 3), activation='relu', padding='same')(encoded)
x = UpSampling2D((2, 2))(x)

# DeConv2
x = Conv2D(32, (3, 3), activation='relu', padding='same')(x)
x = UpSampling2D((2, 2))(x)

# Deconv3
decoded = Conv2D(3, (3, 3), activation='sigmoid', padding='same')(x)

Pour la fonction de perte, on utilise l'**erreur quadratique moyenne (MSE)** entre l'image reconstruite et l'image propre. La `binary_crossentropy`, souvent utilisée avec MNIST, est adaptée à des pixels presque binaires (noir ou blanc), ce qui n'est pas le cas des photographies, dont les pixels prennent toutes les valeurs intermédiaires.

In [ ]:
from keras.models import Model
# Declare the model
autoencoder = Model(input_img, decoded)
autoencoder.compile(optimizer='adam',
                    loss='mse')
autoencoder.summary()

###### Entrainement de l'auto-encodeur

On entraîne l'auto-encodeur avec les constantes définies au début : en entrée les images bruitées, en sortie les images propres.

Le workshop utilisait 100 epochs sur MNIST. Ici, on fixe un maximum de 50 epochs et on ajoute un `EarlyStopping` : l'entraînement s'arrête si la perte de validation ne s'améliore plus pendant 5 epochs, et les meilleurs poids sont conservés. (Le callback TensorBoard du workshop n'est pas repris, la courbe d'apprentissage est tracée directement ci-dessous.)

In [ ]:
# Train the model
history = autoencoder.fit(x_train_noisy, x_train,
                epochs=NB_EPOCHS_DENOISE,
                batch_size=BATCH_SIZE,
                shuffle=True,
                validation_data=(x_val_noisy, x_val),
                callbacks=[keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)]
               )

In [ ]:
# Visualisation des pertes d'apprentissage (Train) et de validation
plt.figure(figsize=(8, 5))
plt.plot(history.history['loss'], label='train')
plt.plot(history.history['val_loss'], label='validation')
plt.title("Loss function evolution (MSE)")
plt.xlabel("Epoch")
plt.ylabel("MSE")
plt.legend()
plt.grid(True)
plt.show()

print(f"final training loss: {history.history['loss'][-1]:.5f}")
print(f"final validation loss: {history.history['val_loss'][-1]:.5f}")

**Que pensez-vous des performances du modèle ?**

*À ajuster selon la courbe obtenue.* On s'attend à ce que les deux courbes diminuent rapidement lors des premières epochs puis se stabilisent. Si la courbe de validation reste proche de celle d'entraînement, le modèle ne fait pas de surapprentissage : il a appris à débruiter de manière générale, et pas seulement les images d'entraînement. Une perte faible ne suffit cependant pas à juger la qualité visuelle : c'est l'objet de l'étude de cas ci-dessous.

## 1.4 Sauvegarde de l'auto-encodeur

In [ ]:
# save the model
autoencoder.save(SAV_MODEL_DENOISE)

# 2. Débruitage des images de test

Débruiter des images revient simplement à faire passer les images bruitées dans l'auto-encodeur entraîné :

In [ ]:
decoded_imgs = autoencoder.predict(x_test_noisy, batch_size=BATCH_SIZE)

In [ ]:
display_image(x_test, x_test_noisy, decoded_imgs, n=8, titles=["Originales", "Bruitées", "Débruitées"])

## 2.1 Mesure de la qualité

Pour juger objectivement les résultats, on compare chaque image (bruitée, puis débruitée) à l'image propre avec deux métriques classiques :

* **PSNR** (en dB) : mesure la fidélité pixel à pixel. Plus il est élevé, plus l'image est proche de l'originale ;
* **SSIM** (entre 0 et 1) : mesure la ressemblance de **structure** (contours, textures). Il est plus proche de la perception humaine et pénalise le flou.

Le débruitage est réussi si les deux scores augmentent nettement entre l'image bruitée et l'image débruitée.

In [ ]:
def psnr(clean, other):
    mse = np.mean((clean - other) ** 2, axis=(1, 2, 3))
    return 10 * np.log10(1.0 / np.maximum(mse, 1e-10))


def ssim(clean, other):
    return np.concatenate([
        tf.image.ssim(tf.constant(clean[i:i + 256]), tf.constant(other[i:i + 256]), max_val=1.0).numpy()
        for i in range(0, len(clean), 256)
    ])


psnr_noisy, ssim_noisy = psnr(x_test, x_test_noisy), ssim(x_test, x_test_noisy)
psnr_denoised, ssim_denoised = psnr(x_test, decoded_imgs), ssim(x_test, decoded_imgs)

print(f"{'':<12}{'PSNR moyen':>12}{'SSIM moyen':>12}")
print(f"{'Bruitées':<12}{psnr_noisy.mean():>9.2f} dB{ssim_noisy.mean():>12.4f}")
print(f"{'Débruitées':<12}{psnr_denoised.mean():>9.2f} dB{ssim_denoised.mean():>12.4f}")

## 2.2 Étude de cas : Originale | Bruitée | Débruitée

On sélectionne quelques images du jeu de test. Pour bien voir la **perte de détails**, on ajoute à droite un **zoom ×3** sur le centre de chaque image : le flou y est beaucoup plus visible qu'à l'échelle de l'image entière. Sous les images bruitées et débruitées sont indiqués le PSNR et le SSIM.

In [ ]:
def zoom(img):
    size = img.shape[0] // 3
    start = (img.shape[0] - size) // 2
    return img[start:start + size, start:start + size]


def case_study(indices):
    cols = ["Originale", "Bruitée", "Débruitée", "Zoom originale", "Zoom bruitée", "Zoom débruitée"]
    fig, axes = plt.subplots(len(indices), len(cols), figsize=(len(cols) * 2.4, len(indices) * 2.6))
    for r, i in enumerate(indices):
        panels = [x_test[i], x_test_noisy[i], decoded_imgs[i]]
        panels += [zoom(p) for p in panels]
        for c, panel in enumerate(panels):
            ax = axes[r, c]
            ax.imshow(np.clip(panel, 0, 1), interpolation="nearest")
            ax.set_xticks([])
            ax.set_yticks([])
            if r == 0:
                ax.set_title(cols[c], fontsize=9)
        axes[r, 1].set_xlabel(f"{psnr_noisy[i]:.1f} dB / {ssim_noisy[i]:.3f}")
        axes[r, 2].set_xlabel(f"{psnr_denoised[i]:.1f} dB / {ssim_denoised[i]:.3f}")
    plt.tight_layout()
    plt.show()


case_study([0, 1, 2, 3, 4])

## 2.3 Le modèle fonctionne-t-il mieux sur certaines images ?

On classe les images de test selon leur SSIM après débruitage et on affiche les 3 meilleures et les 3 moins bonnes reconstructions.

In [ ]:
order = np.argsort(ssim_denoised)

print("Meilleurs cas")
case_study(list(order[-3:][::-1]))
print("Pires cas")
case_study(list(order[:3]))

## 2.4 Application au bruit d'origine des images

Les fichiers du dataset s'appellent `noisy_XXX.jpg` : ils contiennent déjà un léger bruit (grain, artefacts JPEG). On applique ici le modèle **directement sur les images d'origine du jeu de test**, sans ajouter de bruit. On n'a pas d'image propre pour comparer : l'évaluation est uniquement **visuelle**, sur les zooms.

In [ ]:
decoded_orig = autoencoder.predict(x_test, batch_size=BATCH_SIZE)

fig, axes = plt.subplots(4, 4, figsize=(10, 10))
for r, i in enumerate(range(4)):
    for c, (panel, title) in enumerate([(x_test[i], "Image d'origine"), (decoded_orig[i], "Après le modèle"),
                                        (zoom(x_test[i]), "Zoom origine"), (zoom(decoded_orig[i]), "Zoom après le modèle")]):
        axes[r, c].imshow(np.clip(panel, 0, 1), interpolation="nearest")
        axes[r, c].axis("off")
        if r == 0:
            axes[r, c].set_title(title, fontsize=9)
plt.tight_layout()
plt.show()

print(f"écart moyen entre l'image d'origine et la sortie du modèle : PSNR {psnr(x_test, decoded_orig).mean():.2f} dB")

Le modèle a été entraîné à retirer un bruit gaussien de `noise_factor = 0.1`, bien plus fort que le bruit d'origine des fichiers. Sur ces images, on s'attend donc à ce qu'il **lisse** légèrement l'image (le grain et les artefacts JPEG s'atténuent), mais au prix d'un léger flou, puisqu'il « croit » devoir retirer un bruit important. *À vérifier sur les zooms ci-dessus.* Pour traiter ce bruit réel, il faudrait entraîner le modèle avec un bruit plus faible ou de niveau variable.

# 3. Analyse critique des résultats

> Les observations ci-dessous correspondent au comportement attendu de ce type de modèle. **Elles doivent être vérifiées et ajustées à partir des figures et des scores obtenus** lors de l'exécution.

### Le bruit a-t-il bien disparu ?

Sur les zones **uniformes** (ciel, murs, peau, arrière-plans), le grain gaussien disparaît presque complètement : les zooms montrent des aplats lisses. Le PSNR et le SSIM moyens augmentent nettement entre les images bruitées et les images débruitées, ce qui confirme que le modèle a bien appris à retirer ce bruit.

Sur les zones très **texturées**, un léger bruit résiduel peut subsister : pour le modèle, le bruit et une texture fine réelle se ressemblent (ce sont deux variations rapides d'intensité), il a donc du mal à les séparer.

### Certains détails ont-ils été perdus ? L'image est-elle devenue plus floue ?

Oui, c'est la principale limite du modèle :

* les **contours** sont adoucis et les **textures fines** (feuillage, cheveux, herbe, texte présent dans la photo) sont lissées ;
* l'image débruitée est globalement **plus floue** que l'originale, ce qui est très visible dans les zooms ;
* cela s'explique par l'architecture : la résolution est divisée par 4 dans l'encodeur, et `UpSampling2D` se contente de répéter les pixels pour revenir à la taille d'origine. Les détails plus petits que cette échelle ne peuvent pas être parfaitement reconstruits ;
* la perte **MSE** renforce ce phénomène : lorsque le modèle hésite sur la valeur d'un pixel, la MSE le pousse à prédire une **moyenne**, ce qui produit du flou plutôt qu'un détail net ;
* les **couleurs** sont en général bien conservées, puisque le modèle traite les 3 canaux ensemble.

Le **SSIM** est ici plus parlant que le PSNR, car il pénalise directement la perte de structure, donc le flou.

### Le modèle fonctionne-t-il mieux sur certaines images que sur d'autres ?

Oui :

* **meilleurs cas** : images avec de grandes zones homogènes et peu de textures (fond uni, ciel, objets simples). Il y a peu de détails à perdre, et le bruit y est facile à repérer ;
* **pires cas** : images très détaillées ou chargées (végétation dense, foule, scènes urbaines), où le modèle lisse à la fois le bruit et les détails utiles.

### Limites de la démarche

* **Bruit synthétique** : le modèle n'a vu qu'un seul type de bruit (gaussien, `noise_factor = 0.1`). Le bruit réel de numérisation (grain du papier, compression JPEG, poussières, rayures) est différent, et le modèle risque d'être moins efficace dessus.
* **Bruit fixe** : comme dans le workshop, le bruit d'entraînement est généré une seule fois. Le modèle voit donc toujours le même bruit sur une même image ; régénérer le bruit à chaque epoch augmenterait la diversité des exemples.
* **Taille du dataset** : 148 images, c'est très peu. Le découpage en patchs compense en partie, mais les patchs d'une même image se ressemblent : le modèle voit peu de types de scènes différents, et les scores du test (une vingtaine d'images) sont peu précis.
* **Référence imparfaite** : les images de référence contiennent déjà un léger bruit d'origine. Le modèle apprend donc à reproduire ce léger bruit, et les scores PSNR / SSIM mesurent l'écart à ces images, pas à des images parfaitement propres.
* **Résolution** : travailler en 256×256 limite le temps de calcul, mais une partie des détails fins est déjà perdue au redimensionnement.

### Pistes d'amélioration

* ajouter des **connexions directes** (*skip connections*, type U-Net) entre l'encodeur et le décodeur, pour transmettre les détails haute résolution au décodeur et réduire le flou ;
* remplacer `UpSampling2D` par des convolutions transposées (`Conv2DTranspose`), qui apprennent à agrandir l'image ;
* utiliser une perte qui pénalise moins le détail (L1, SSIM) ;
* entraîner avec plusieurs niveaux et types de bruit pour gagner en robustesse.

### Conclusion

L'auto-encodeur convolutif du workshop, adapté aux images RGB et entraîné sur des patchs extraits des 148 photos du dataset, permet de supprimer efficacement un bruit gaussien sur des photographies, avec un gain mesurable en PSNR et SSIM. Sa principale faiblesse est la **perte de détails fins**, qui rend les images légèrement plus floues, en particulier les plus texturées. Pour l'étape suivante du projet (génération de légendes), ce léger flou est probablement acceptable tant que les objets restent reconnaissables, mais il faudrait entraîner le modèle sur un bruit plus proche des défauts réels de numérisation.